<a href="https://colab.research.google.com/github/AustinBramlett/movie-recommender-svd/blob/main/notebooks/01_ratings_matrix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [31]:
movies = ["Action 1", "Action 2", "Action 3",
          "Romance 1", "Romance 2", "Romance 3"]

users = ["Ana", "Ben", "Cal", "Dee", "Eve", "Fay"]

In [32]:
toy = np.array([
    [5, 4, 5, 1, np.nan, 2],  # Ana
    [4, np.nan, 5, 2, 1, 1],  # Ben
    [5, 5, np.nan, 1, 2, np.nan],  # Cal
    [1, 2, 1, 5, 4, np.nan],  # Dee
    [np.nan, 1, 2, 4, 5, 5],  # Eve
    [2, 1, np.nan, 5, np.nan, 4]  # Fay
])

In [33]:
print(toy_table)

     Action 1  Action 2  Action 3  Romance 1  Romance 2  Romance 3
Ana       5.0       4.0       5.0        1.0        NaN        2.0
Ben       4.0       NaN       5.0        2.0        1.0        1.0
Cal       5.0       5.0       NaN        1.0        2.0        NaN
Dee       1.0       2.0       1.0        5.0        4.0        NaN
Eve       NaN       1.0       2.0        4.0        5.0        5.0
Fay       2.0       1.0       NaN        5.0        NaN        4.0


In [34]:
missing = np.isnan(toy).sum()
total = toy.size

print("Missing ratings:", missing)
print("Total ratings:", total)
print("Percentage missing:", missing / total * 100)

Missing ratings: 8
Total ratings: 36
Percentage missing: 22.22222222222222


In [35]:
movie_averages = np.nanmean(toy, axis=0)

for i in range(len(movies)):
    print(movies[i], ":", round(movie_averages[i], 2))

Action 1 : 3.4
Action 2 : 2.6
Action 3 : 3.25
Romance 1 : 3.0
Romance 2 : 3.0
Romance 3 : 3.0


In [36]:
toy_filled = toy.copy()

for j in range(toy.shape[1]):
    toy_filled[np.isnan(toy_filled[:, j]), j] = movie_averages[j]

print(pd.DataFrame(toy_filled, index=users, columns=movies))

     Action 1  Action 2  Action 3  Romance 1  Romance 2  Romance 3
Ana       5.0       4.0      5.00        1.0        3.0        2.0
Ben       4.0       2.6      5.00        2.0        1.0        1.0
Cal       5.0       5.0      3.25        1.0        2.0        3.0
Dee       1.0       2.0      1.00        5.0        4.0        3.0
Eve       3.4       1.0      2.00        4.0        5.0        5.0
Fay       2.0       1.0      3.25        5.0        3.0        4.0


In [37]:
import urllib.request
import zipfile
import io

url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"

data = urllib.request.urlopen(url).read()
zip_file = zipfile.ZipFile(io.BytesIO(data))

ratings = pd.read_csv(
    zip_file.open("ml-100k/u.data"),
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

print(ratings.head())

   user_id  movie_id  rating  timestamp
0      196       242       3  881250949
1      186       302       3  891717742
2       22       377       1  878887116
3      244        51       2  880606923
4      166       346       1  886397596


In [38]:
print("Number of ratings:", len(ratings))
print("Number of users:", ratings["user_id"].nunique())
print("Number of movies:", ratings["movie_id"].nunique())

Number of ratings: 100000
Number of users: 943
Number of movies: 1682


In [39]:
average_rating = ratings["rating"].mean()

print("Average rating:", round(average_rating, 2))

Average rating: 3.53


In [40]:
user_movie_matrix = ratings.pivot(
    index="user_id",
    columns="movie_id",
    values="rating"
)

print("Matrix dimensions:", user_movie_matrix.shape)
print(user_movie_matrix.head())

Matrix dimensions: (943, 1682)
movie_id  1     2     3     4     5     6     7     8     9     10    ...  \
user_id                                                               ...   
1          5.0   3.0   4.0   3.0   3.0   5.0   4.0   1.0   5.0   3.0  ...   
2          4.0   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   2.0  ...   
3          NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
4          NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
5          4.0   3.0   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   

movie_id  1673  1674  1675  1676  1677  1678  1679  1680  1681  1682  
user_id                                                               
1          NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  
2          NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  
3          NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  
4          NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  
5  

In [41]:
missing_count = user_movie_matrix.isna().sum().sum()
total_count = user_movie_matrix.size
missing_percentage = missing_count / total_count * 100

print("Missing ratings:", missing_count)
print("Total entries:", total_count)
print("Percentage missing:", round(missing_percentage, 2), "%")

Missing ratings: 1486126
Total entries: 1586126
Percentage missing: 93.7 %


In [42]:
movie_means = user_movie_matrix.mean(axis=0)

print("First 10 movie averages:")
print(movie_means.head(10))

First 10 movie averages:
movie_id
1     3.878319
2     3.206107
3     3.033333
4     3.550239
5     3.302326
6     3.576923
7     3.798469
8     3.995434
9     3.896321
10    3.831461
dtype: float64


In [43]:
movie_counts = user_movie_matrix.count(axis=0)
top_5 = movie_counts.sort_values(ascending=False).head(5)

print("Top 5 most-rated movies:")
print(top_5)

Top 5 most-rated movies:
movie_id
50     583
258    509
100    508
181    507
294    485
dtype: int64


In [44]:
print("Missing ratings:")
for i in range(len(users)):
    for j in range(len(movies)):
        if np.isnan(toy[i, j]):
            print(users[i], "-", movies[j])

Missing ratings:
Ana - Romance 2
Ben - Action 2
Cal - Action 3
Cal - Romance 3
Dee - Romance 3
Eve - Action 1
Fay - Action 3
Fay - Romance 2


In [45]:
predictions = [
    ("Ana", "Romance 2", 3.0),
    ("Ben", "Action 2", 3.0),
    ("Cal", "Action 3", 3.25),
    ("Cal", "Romance 3", 3.0),
    ("Dee", "Romance 3", 3.0),
    ("Eve", "Action 1", 3.4),
    ("Fay", "Action 3", 3.25),
    ("Fay", "Romance 2", 3.0)
]
[45]
0s

for user, movie, rating in predictions:
    print(user, "-", movie, ":", rating, "stars")
Ana - Romance 2 : 3.0 stars
Ben - Action 2 : 3.0 stars
Cal - Action 3 : 3.25 stars
Cal - Romance 3 : 3.0 stars
Dee - Romance 3 : 3.0 stars
Eve - Action 1 : 3.4 stars
Fay - Action 3 : 3.25 stars
Fay - Romance 2 : 3.0 stars$0
for user, movie, rating in predictions:
    print(user, "-", movie, ":", rating, "stars")

Ana - Romance 2 : 3.0 stars
Ben - Action 2 : 3.0 stars
Cal - Action 3 : 3.25 stars
Cal - Romance 3 : 3.0 stars
Dee - Romance 3 : 3.0 stars
Eve - Action 1 : 3.4 stars
Fay - Action 3 : 3.25 stars
Fay - Romance 2 : 3.0 stars
